# Fase 1 · Paso 2: descarga selectiva de un lote

Esta libreta prepara una lista exacta de rutas S3, utiliza el cliente oficial de NDA, audita todos los paquetes en almacenamiento temporal y solo después copia el lote íntegro y válido a Drive. No entrena modelos ni abre la prueba reservada.

## 1. Montar Drive y localizar la raíz privada

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
data_root = Path(private_root)
if not data_root.is_dir():
    raise FileNotFoundError('La raíz privada configurada no está disponible.')
print('Drive montado y raíz privada localizada.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import shutil
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_descarga')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

Se fija `nda-tools==0.7.0`, versión oficial publicada en PyPI.

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nda-tools==0.7.0', 'keyrings.alt'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Cliente oficial y pruebas automáticas preparados.')

## 4. Elegir y preparar el lote

Para la primera ejecución escribe `1`. Cada lote se determina desde la cola privada; no se seleccionan rutas manualmente.

In [ ]:
inventory_config = REPO_DIR / 'configs' / 'acquisition_inventory.example.json'
subprocess.run([sys.executable, '-m', 'knee.acquisition_inventory', '--config', str(inventory_config)], cwd=REPO_DIR, check=True)
frozen_plan = data_root / 'outputs' / 'auditorias' / 'fase_1' / 'paso_2_inventario' / 'plan_descarga_congelado_privado.csv'
if not frozen_plan.is_file():
    raise FileNotFoundError('No se creó el plan congelado de descarga.')
print('Asignación adquisición-lote congelada antes de iniciar la transferencia.')
batch_number = int(input('Número de lote a descargar (1-20): ').strip())
if not 1 <= batch_number <= 20:
    raise ValueError('El lote debe estar entre 1 y 20.')
queue_path = data_root / 'outputs' / 'auditorias' / 'fase_1' / 'paso_2_inventario' / 'cola_descarga_selectiva_privada.csv'
if not queue_path.is_file():
    raise FileNotFoundError('No se encontró la cola privada del inventario.')
work_root = Path(f'/content/knee_oai_lote_{batch_number:03d}')
if work_root.exists():
    raise RuntimeError('La carpeta temporal del lote ya existe; usa un entorno nuevo para evitar mezclar ejecuciones.')
download_dir = work_root / 'descarga'
download_dir.mkdir(parents=True)
batch_txt = work_root / f'lote_{batch_number:03d}_s3.txt'
subprocess.run([sys.executable, '-m', 'knee.selective_download', 'prepare', '--queue', str(queue_path), '--batch', str(batch_number), '--output', str(batch_txt)], cwd=REPO_DIR, check=True)
print('Lista exacta del lote preparada sin mostrar rutas privadas.')

## 5. Autenticar y descargar con el cliente oficial de NDA

Configura `NDA_USERNAME` y `NDA_PACKAGE_ID` como secretos de Colab. La contraseña se solicita de forma oculta y se guarda únicamente en un almacén temporal del entorno; nunca se escribe en Drive ni en el repositorio.

In [ ]:
from getpass import getpass

nda_username = userdata.get('NDA_USERNAME')
nda_package_id = userdata.get('NDA_PACKAGE_ID')
if not nda_username or not nda_package_id:
    raise RuntimeError('Configura los secretos NDA_USERNAME y NDA_PACKAGE_ID en Colab.')
if not str(nda_package_id).isdigit():
    raise ValueError('NDA_PACKAGE_ID debe ser numérico.')
keyring_root = Path('/content/.knee_nda_keyring')
os.environ['XDG_DATA_HOME'] = str(keyring_root)
os.environ['PYTHON_KEYRING_BACKEND'] = 'keyrings.alt.file.PlaintextKeyring'
import keyring

nda_password = getpass('Contraseña de la cuenta NDA: ')
keyring.set_password('nda-tools', nda_username, nda_password)
del nda_password
nda_log_dir = work_root / 'logs'
nda_log_dir.mkdir()
downloadcmd = shutil.which('downloadcmd')
if not downloadcmd:
    raise RuntimeError('No se encontró el cliente oficial downloadcmd.')
try:
    completed = subprocess.run([downloadcmd, '-dp', str(nda_package_id), '-t', str(batch_txt), '-d', str(download_dir), '-wt', '4', '-u', nda_username, '--log-dir', str(nda_log_dir)])
    if completed.returncode != 0:
        raise RuntimeError(f'NDA finalizó con código {completed.returncode}; conserva el entorno para revisar los logs.')
finally:
    try:
        keyring.delete_password('nda-tools', nda_username)
    except Exception:
        pass
    shutil.rmtree(keyring_root, ignore_errors=True)
print('Descarga oficial terminada; credencial temporal eliminada.')

## 6. Auditar el lote y promoverlo a Drive

Nada se copia a Drive si falta un archivo, aparece un paquete inesperado, existe un duplicado o un DICOM no puede decodificarse.

In [ ]:
destination_root = data_root / 'dicom' / 'originales' / 'cohorte_v00'
evidence_dir = data_root / 'outputs' / 'auditorias' / 'fase_1' / 'paso_2_inventario' / 'lotes'
subprocess.run([sys.executable, '-m', 'knee.selective_download', 'promote', '--download-dir', str(download_dir), '--batch-file', str(batch_txt), '--destination-root', str(destination_root), '--batch', str(batch_number), '--evidence-dir', str(evidence_dir)], cwd=REPO_DIR, check=True)
print('Lote íntegro, legible y verificado por contenido; copia canónica guardada en Drive.')

## 7. Actualizar el inventario general y limpiar el espacio temporal

In [ ]:
import json
from datetime import datetime, timezone

subprocess.run([sys.executable, '-m', 'knee.acquisition_inventory', '--config', str(inventory_config)], cwd=REPO_DIR, check=True)
summary_path = data_root / 'outputs' / 'auditorias' / 'fase_1' / 'paso_2_inventario' / 'resumen_inventario_publico.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
record = {
    'phase': 1,
    'step': 2,
    'operation': 'selective_download_batch',
    'batch_number': batch_number,
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'inventory_status': summary['status'],
    'local_status_counts': summary['local_status_counts'],
    'pending_download_or_replacement': summary['pending_download_or_replacement'],
    'training_executed': False,
    'reserved_test_opened': False,
}
record_path = evidence_dir / f'lote_{batch_number:03d}_registro_ejecucion.json'
record_path.write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
shutil.rmtree(work_root)
print('Lote finalizado, inventario actualizado y espacio temporal eliminado.', record)